In [0]:
# DATA INGESTION (Bronze Layer)
df = spark.read.csv("/Workspace/Users/aakankshapednekar19@gmail.com/sales_data.csv",
                       header= True ,
                       inferSchema= True)
df.show()

+-----------+---------------+---------+---------------+-------+---------------+-------+------+--------+-------+-----------+----+-----------+--------------------+----------------+--------------------+------------+-------------+--------+----------+---------+---------+---------------+----------------+--------+
|ORDERNUMBER|QUANTITYORDERED|PRICEEACH|ORDERLINENUMBER|  SALES|      ORDERDATE| STATUS|QTR_ID|MONTH_ID|YEAR_ID|PRODUCTLINE|MSRP|PRODUCTCODE|        CUSTOMERNAME|           PHONE|        ADDRESSLINE1|ADDRESSLINE2|         CITY|   STATE|POSTALCODE|  COUNTRY|TERRITORY|CONTACTLASTNAME|CONTACTFIRSTNAME|DEALSIZE|
+-----------+---------------+---------+---------------+-------+---------------+-------+------+--------+-------+-----------+----+-----------+--------------------+----------------+--------------------+------------+-------------+--------+----------+---------+---------+---------------+----------------+--------+
|      10107|             30|     95.7|              2| 2871.0| 2/24/2003

In [0]:
df.printSchema()

root
 |-- ORDERNUMBER: integer (nullable = true)
 |-- QUANTITYORDERED: integer (nullable = true)
 |-- PRICEEACH: double (nullable = true)
 |-- ORDERLINENUMBER: integer (nullable = true)
 |-- SALES: double (nullable = true)
 |-- ORDERDATE: string (nullable = true)
 |-- STATUS: string (nullable = true)
 |-- QTR_ID: integer (nullable = true)
 |-- MONTH_ID: integer (nullable = true)
 |-- YEAR_ID: integer (nullable = true)
 |-- PRODUCTLINE: string (nullable = true)
 |-- MSRP: integer (nullable = true)
 |-- PRODUCTCODE: string (nullable = true)
 |-- CUSTOMERNAME: string (nullable = true)
 |-- PHONE: string (nullable = true)
 |-- ADDRESSLINE1: string (nullable = true)
 |-- ADDRESSLINE2: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- POSTALCODE: string (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- TERRITORY: string (nullable = true)
 |-- CONTACTLASTNAME: string (nullable = true)
 |-- CONTACTFIRSTNAME: string (nullable = tr

In [0]:
# Save as Bronze
df_raw = df
df_raw.write.format("delta").mode("overwrite").option("delta.columnMapping.mode","name").saveAsTable("sales_bronze")

In [0]:
# Data Cleaning (Silver Layer)
df = spark.read.table("sales_bronze")
df.printSchema()

root
 |-- ORDERNUMBER: integer (nullable = true)
 |-- QUANTITYORDERED: integer (nullable = true)
 |-- PRICEEACH: double (nullable = true)
 |-- ORDERLINENUMBER: integer (nullable = true)
 |-- SALES: double (nullable = true)
 |-- ORDERDATE: string (nullable = true)
 |-- STATUS: string (nullable = true)
 |-- QTR_ID: integer (nullable = true)
 |-- MONTH_ID: integer (nullable = true)
 |-- YEAR_ID: integer (nullable = true)
 |-- PRODUCTLINE: string (nullable = true)
 |-- MSRP: integer (nullable = true)
 |-- PRODUCTCODE: string (nullable = true)
 |-- CUSTOMERNAME: string (nullable = true)
 |-- PHONE: string (nullable = true)
 |-- ADDRESSLINE1: string (nullable = true)
 |-- ADDRESSLINE2: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- POSTALCODE: string (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- TERRITORY: string (nullable = true)
 |-- CONTACTLASTNAME: string (nullable = true)
 |-- CONTACTFIRSTNAME: string (nullable = tr

In [0]:
# Feature Engineering - Rename Columns
df = df.toDF(*[col.lower() for col in df.columns])
df.printSchema()

root
 |-- ordernumber: integer (nullable = true)
 |-- quantityordered: integer (nullable = true)
 |-- priceeach: double (nullable = true)
 |-- orderlinenumber: integer (nullable = true)
 |-- sales: double (nullable = true)
 |-- orderdate: string (nullable = true)
 |-- status: string (nullable = true)
 |-- qtr_id: integer (nullable = true)
 |-- month_id: integer (nullable = true)
 |-- year_id: integer (nullable = true)
 |-- productline: string (nullable = true)
 |-- msrp: integer (nullable = true)
 |-- productcode: string (nullable = true)
 |-- customername: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- addressline1: string (nullable = true)
 |-- addressline2: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postalcode: string (nullable = true)
 |-- country: string (nullable = true)
 |-- territory: string (nullable = true)
 |-- contactlastname: string (nullable = true)
 |-- contactfirstname: string (nullable = tr

In [0]:
# Handle Missing Values
from pyspark.sql.functions import col, when

df = df.withColumn(
    "state",
    when(col("state").isNull(), "NA").otherwise(col("state"))
)

In [0]:
# Drop Useless Columns
df = df.drop(
    "addressline1",
    "addressline2",
    "postalcode",
    "phone",
)
df.printSchema()

root
 |-- ordernumber: integer (nullable = true)
 |-- quantityordered: integer (nullable = true)
 |-- priceeach: double (nullable = true)
 |-- orderlinenumber: integer (nullable = true)
 |-- sales: double (nullable = true)
 |-- orderdate: string (nullable = true)
 |-- status: string (nullable = true)
 |-- qtr_id: integer (nullable = true)
 |-- month_id: integer (nullable = true)
 |-- year_id: integer (nullable = true)
 |-- productline: string (nullable = true)
 |-- msrp: integer (nullable = true)
 |-- productcode: string (nullable = true)
 |-- customername: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- territory: string (nullable = true)
 |-- contactlastname: string (nullable = true)
 |-- contactfirstname: string (nullable = true)
 |-- dealsize: string (nullable = true)



In [0]:
# Converting Mixed Date-Time format to One Standard Format
from pyspark.sql.functions import expr

df = df.withColumn(
    "orderdate",
    expr("to_timestamp(orderdate, 'M/d/yyyy H:mm')")
)
df.show(5)

+-----------+---------------+---------+---------------+-------+-------------------+-------+------+--------+-------+-----------+----+-----------+--------------------+-------------+-----+-------+---------+---------------+----------------+--------+
|ordernumber|quantityordered|priceeach|orderlinenumber|  sales|          orderdate| status|qtr_id|month_id|year_id|productline|msrp|productcode|        customername|         city|state|country|territory|contactlastname|contactfirstname|dealsize|
+-----------+---------------+---------+---------------+-------+-------------------+-------+------+--------+-------+-----------+----+-----------+--------------------+-------------+-----+-------+---------+---------------+----------------+--------+
|      10107|             30|     95.7|              2| 2871.0|2003-02-24 00:00:00|Shipped|     1|       2|   2003|Motorcycles|  95|   S10_1678|   Land of Toys Inc.|          NYC|   NY|    USA|       NA|             Yu|            Kwai|   Small|
|      10121|   

In [0]:
(df.count(), len(df.columns))

(2823, 21)

In [0]:
df = df.dropna()
(df.count(), len(df.columns))

(2823, 21)

In [0]:
df = df.dropDuplicates()
(df.count(), len(df.columns))

(2823, 21)

In [0]:
# Save Silver
df.write.format("delta").mode("overwrite").saveAsTable("sales_silver")

In [0]:
# Gold Layer - Sales by Country
spark.sql("""
CREATE OR REPLACE TABLE sales_gold_country AS
SELECT
    country,
    SUM(sales) AS total_sales,
    COUNT(ordernumber) AS total_orders
FROM sales_silver
GROUP BY country
""")

spark.sql("SELECT * FROM sales_gold_country").display()

country,total_sales,total_orders
Australia,630623.0999999999,185
Sweden,210014.20999999996,57
Denmark,245637.15000000005,63
Switzerland,117713.55999999997,31
Austria,202062.52999999997,55
Spain,1215686.9200000009,342
Philippines,94015.72999999998,26
Japan,188167.80999999997,52
Belgium,108412.61999999998,33
USA,3627982.83,1004


Databricks visualization. Run in Databricks to view.

In [0]:
# Sales by Product Line
spark.sql("""
CREATE OR REPLACE TABLE sales_gold_product AS
SELECT
    productline,
    SUM(sales) AS total_sales,
    AVG(priceeach) AS avg_price
FROM sales_silver
GROUP BY productline
""")

spark.sql("SELECT * FROM sales_gold_product").display()

productline,total_sales,avg_price
Classic Cars,3919615.660000001,87.33578076525336
Ships,714437.13,83.85547008547006
Trucks and Buses,1127789.8399999996,87.52794019933553
Trains,226243.47000000006,75.65467532467534
Vintage Cars,1903150.8399999987,78.14820428336084
Planes,975003.5699999996,81.74091503267974
Motorcycles,1166388.3400000005,82.99755287009064


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# Monthly Sales
spark.sql("""
CREATE OR REPLACE TABLE sales_gold_monthly AS
SELECT
    year(orderdate) AS year,
    month(orderdate) AS month,
    SUM(sales) AS monthly_sales
FROM sales_silver
GROUP BY year(orderdate), month(orderdate)
ORDER BY year(orderdate) , month(orderdate)
""")


spark.sql("SELECT * FROM sales_gold_monthly").display()

year,month,monthly_sales
2003,1,129753.59999999998
2003,2,140836.19000000003
2003,3,174504.9
2003,4,201609.55000000005
2003,5,192673.11
2003,6,168082.56
2003,7,187731.88
2003,8,197809.29999999996
2003,9,263973.36000000004
2003,10,568290.9700000002


Databricks visualization. Run in Databricks to view.

In [0]:
#  Top Customers
spark.sql("""
CREATE OR REPLACE TABLE sales_gold_customers AS
SELECT
    customername,
    SUM(sales) AS total_sales,
    COUNT(ordernumber) AS total_orders
FROM sales_silver
GROUP BY customername
""")

spark.sql("SELECT * FROM sales_gold_customers").display()

customername,total_sales,total_orders
Reims Collectables,135042.94,41
Super Scale Inc.,79472.07,17
Suominen Souveniers,113961.14999999998,30
Vitachrome Inc.,88041.26000000002,25
"Saveley & Henriot, Co.",142874.25,41
Enaco Distributors,78411.85999999999,23
"Volvo Model Replicas, Co",75754.87999999999,19
Scandinavian Gift Ideas,134259.33,38
"Corrida Auto Replicas, Ltd",120615.28,32
Canadian Gift Exchange Network,75238.92000000001,22


In [0]:
# Partitioned by Country
df.write.format("delta") \
  .partitionBy("country") \
  .mode("overwrite") \
  .saveAsTable("sales_silver_partitioned")

spark.sql("SELECT * FROM sales_silver_partitioned").display()

ordernumber,quantityordered,priceeach,orderlinenumber,sales,orderdate,status,qtr_id,month_id,year_id,productline,msrp,productcode,customername,city,state,country,territory,contactlastname,contactfirstname,dealsize
10183,23,100.0,8,5372.57,2003-11-13T00:00:00.000Z,Shipped,4,11,2003,Classic Cars,214,S10_1949,"Classic Gift Ideas, Inc",Philadelphia,PA,USA,NA,Cervantes,Francisca,Medium
10369,41,100.0,2,4514.92,2005-01-20T00:00:00.000Z,Shipped,1,1,2005,Classic Cars,214,S10_1949,Collectables For Less Inc.,Brickhaven,MA,USA,NA,Nelson,Allen,Medium
10145,37,100.0,9,5192.95,2003-08-25T00:00:00.000Z,Shipped,3,8,2003,Motorcycles,118,S10_2016,Toys4GrownUps.com,Pasadena,CA,USA,NA,Young,Julie,Medium
10159,22,100.0,16,4132.7,2003-10-10T00:00:00.000Z,Shipped,4,10,2003,Motorcycles,193,S10_4698,Corporate Gift Ideas Co.,San Francisco,CA,USA,NA,Brown,Julie,Medium
10245,28,100.0,2,4591.72,2004-05-04T00:00:00.000Z,Shipped,2,5,2004,Classic Cars,147,S10_4962,Super Scale Inc.,New Haven,CT,USA,NA,Murphy,Leslie,Medium
10245,38,100.0,6,5920.4,2004-05-04T00:00:00.000Z,Shipped,2,5,2004,Trucks and Buses,136,S12_1666,Super Scale Inc.,New Haven,CT,USA,NA,Murphy,Leslie,Medium
10333,33,99.21,6,3273.93,2004-11-18T00:00:00.000Z,Shipped,4,11,2004,Trucks and Buses,136,S12_1666,Mini Wheels Co.,San Francisco,CA,USA,NA,Murphy,Julie,Medium
10201,25,100.0,1,4029.0,2003-12-01T00:00:00.000Z,Shipped,4,12,2003,Motorcycles,150,S12_2823,Mini Wheels Co.,San Francisco,CA,USA,NA,Murphy,Julie,Medium
10272,27,100.0,3,4283.01,2004-07-20T00:00:00.000Z,Shipped,3,7,2004,Classic Cars,151,S12_3148,Diecast Classics Inc.,Allentown,PA,USA,NA,Yu,Kyung,Medium
10127,42,100.0,1,8138.76,2003-06-03T00:00:00.000Z,Shipped,2,6,2003,Classic Cars,173,S12_3891,Muscle Machine Inc,NYC,NY,USA,NA,Young,Jeff,Large
